# BLOQUE 02 – Diagnóstico de mercado y demanda
## Objetivo
Integrar las capas de la previa etapa en un solo geopackage, y realizar el filtrado de lo que se necesitará en la etapa 3.
## Inputs
- 1_D2C1_Poblacion.gpkg
- 1_D2C2_Económico.gpkg
## Procesos
- Integración
- Filtrado
## Output
- 2_Dimensiones.gpkg
    - 2_D2_mercado_demanda.shp
      

In [1]:
ruta_poblacion = r"C:\Users\alflo\Desktop\DAL_LAB\outputs\etapa_1\1_D2C1_Poblacion.gpkg"
ruta_economico = r"C:\Users\alflo\Desktop\DAL_LAB\outputs\etapa_1\1_D2C2_Económico.gpkg"

ruta_salida = r"C:\Users\alflo\Desktop\DAL_LAB\outputs\etapa_2\2_Dimensiones.gpkg"

# Carga de datos

In [2]:
import pandas as pd
import geopandas as gpd
import numpy as np

In [3]:
gpd.list_layers(ruta_poblacion)

,name,geometry_type
0,1_D2C1V1_Pop_tot_obj,Polygon
1,1_D2C1V2_Condicionantes,Polygon


In [4]:
pob_tot = gpd.read_file(ruta_poblacion, layer = "1_D2C1V1_Pop_tot_obj")
condicion = gpd.read_file(ruta_poblacion, layer = "1_D2C1V2_Condicionantes")
empleos = gpd.read_file(ruta_economico, layer = "1_D2C2V1_Mzn_Empleos")

In [5]:
pob_tot.columns

Index(['CVEGEO', 'CVE_ENT', 'CVE_MUN', 'CVE_LOC', 'CVE_AGEB', 'CVE_MZA',
       'AMBITO', 'TIPOMZA', 'POBTOT', 'POBFEM', 'POBMAS', 'P_0A2', 'P_0A2_F',
       'P_0A2_M', 'P_3YMAS', 'P_3YMAS_F', 'P_3YMAS_M', 'P_5YMAS', 'P_5YMAS_F',
       'P_5YMAS_M', 'P_12YMAS', 'P_12YMAS_F', 'P_12YMAS_M', 'P_15YMAS',
       'P_15YMAS_F', 'P_15YMAS_M', 'P_18YMAS', 'P_18YMAS_F', 'P_18YMAS_M',
       'P_3A5', 'P_3A5_F', 'P_3A5_M', 'P_6A11', 'P_6A11_F', 'P_6A11_M',
       'P_8A14', 'P_8A14_F', 'P_8A14_M', 'P_12A14', 'P_12A14_F', 'P_12A14_M',
       'P_15A17', 'P_15A17_F', 'P_15A17_M', 'P_18A24', 'P_18A24_F',
       'P_18A24_M', 'P_15A49_F', 'P_60YMAS', 'P_60YMAS_F', 'P_60YMAS_M',
       'POB0_14', 'POB15_64', 'POB65_MAS', 'geometry'],
      dtype='object')

In [6]:
condicion.columns

Index(['CVEGEO', 'CVE_ENT', 'CVE_MUN', 'CVE_LOC', 'CVE_AGEB', 'CVE_MZA',
       'AMBITO', 'TIPOMZA', 'PCON_DISC', 'PCDISC_MOT', 'PCDISC_VIS',
       'PCDISC_LENG', 'PCDISC_AUD', 'PCDISC_MOT2', 'PCDISC_MEN', 'PCON_LIMI',
       'PCLIM_CSB', 'PCLIM_VIS', 'PCLIM_HACO', 'PCLIM_OAUD', 'PCLIM_MOT2',
       'PCLIM_RE_CO', 'PCLIM_PMEN', 'PSIND_LIM', 'CVEAGEB', 'REZAGO',
       'geometry'],
      dtype='object')

In [7]:
empleos.columns

Index(['CVEGEO', 'CVE_ENT', 'CVE_MUN', 'CVE_LOC', 'CVE_AGEB', 'CVE_MZA',
       'AMBITO', 'TIPOMZA', 'Emp', 'geometry'],
      dtype='object')

# Unión a un solo df, extracción de columnas de interés

In [8]:
# Población
cols_pob = ["CVEGEO", 'CVE_ENT', 'CVE_MUN', 'CVE_LOC', 'CVE_AGEB', 'CVE_MZA', 'AMBITO', 'TIPOMZA',
            'POBTOT',  'POBFEM', 'POBMAS', 'P_60YMAS', 'P_60YMAS_F', 'P_60YMAS_M',
            'geometry']
cols_cond = ["CVEGEO", 'PCON_DISC', 'PCDISC_MOT', 'PCDISC_MOT2', 'PCON_LIMI', 'PCLIM_CSB', 'PCLIM_MOT2']
cols_emp = ["CVEGEO", "Emp"]

pob_tot = pob_tot[cols_pob]
condicion = condicion[cols_cond]
empleos = empleos[cols_emp]

In [9]:
#Juntar df's
df = pob_tot.merge(condicion, on="CVEGEO", how="left")
df = df.merge(empleos, on="CVEGEO", how="left")

df['area_ha'] = df.area / 10000

# filtrado estado y municipios

In [10]:
# EXTRAER JALISCO
df_ent = df[df['CVE_ENT'] == '14'].copy()

# Lista de municipios AMG
municipios = ['002','039', '044', '051','070','097', '098', '101', '120', '124']

# Filtrar el DataFrame solo con los municipios de interés
df_mun = df_ent[
    (df_ent['CVE_MUN'].isin(municipios)) & 
    (df_ent['CVE_ENT'] == '14')
]

# Limpieza asteriscos y n/d

In [11]:
# Reemplazar * con 1
def replace(dataframe):
    return dataframe.replace('*', '1')

# Limpiar *
df_mun = replace(df_mun)

# Reemplazar N/D
df_mun = df_mun.replace('N/D', np.nan)

# Actualizar valores sin información de "null" a 0
df_mun = df_mun.fillna(0)

# Limpiar duplicados

In [12]:
#Checar por duplicados
# Detectar duplicados basados en el identificador CVEGEO
duplicados = df_mun[df_mun.duplicated(subset="CVEGEO", keep=False)]

if not duplicados.empty:
    print("Se encontraron manzanas duplicadas:")
    print(duplicados.shape)

    # Eliminar duplicados, conservando el primero que aparece
    df_mun = df_mun.drop_duplicates(subset="CVEGEO", keep="first")
    print(f"Se han eliminado los duplicados. Total de registros ahora: {len(df_mun)}")
else:
    print("No se encontraron manzanas duplicadas.")


No se encontraron manzanas duplicadas.


# Guardado 

In [13]:
df_mun.to_file(ruta_salida, layer='2_D2_mercado_demanda', driver='GPKG')